# ReAct RAG with AWS

## 📖 Overview

This notebook demonstrates **ReAct (Reason + Act) RAG** using AWS services:
- **Interleaved reasoning and action** cycles
- **Thought**: LLM reasons about what to do next
- **Action**: Execute tools (retrieve, search, calculate)
- **Observation**: Process action results
- **Repeat**: Continue until answer is ready

### What is ReAct RAG?

**Traditional RAG (Fixed Pipeline):**
```
Query → Retrieve → Generate → Done
│
└─ No flexibility! Fixed sequence
```

**ReAct RAG (Reason + Act Loop):**
```
Query → Think: "What do I need?"
         ↓
         Act: Retrieve/Search/Calculate
         ↓
         Observe: "What did I learn?"
         ↓
         Think: "Is this enough? What next?"
         ↓
         Act: Another action OR Generate answer
         ↓
         Repeat...
```

### Key Concepts

1. **Thought**: LLM reasons explicitly
   - "I need to find pricing information"
   - "I should compare these options"
   - "I have enough info to answer"

2. **Action**: Execute a tool/operation
   - Retrieve documents
   - Search knowledge base
   - Calculate numbers
   - Call external APIs

3. **Observation**: Process results
   - What did the action return?
   - Is it helpful?
   - What's still missing?

4. **Loop**: Continue until done
   - Not fixed number of steps
   - Adapts to query complexity
   - Stops when ready

### Why ReAct?

**Problems it Solves:**
- ❌ Fixed pipeline can't adapt
- ❌ No way to course-correct
- ❌ Can't decide what to do next
- ❌ Limited to pre-defined sequences
- ❌ Can't handle dynamic needs

**ReAct Solutions:**
- ✅ Adaptive: decides next action based on state
- ✅ Course-correction: can change approach
- ✅ Autonomous: self-directed reasoning
- ✅ Flexible: not bound to fixed sequence
- ✅ Dynamic: handles varying complexity

### When to Use

✅ **Good for:**
- Uncertain retrieval needs
- Need multiple tool types
- Complex multi-step tasks
- Require adaptive behavior
- Interactive problem solving
- Research and exploration

❌ **Not ideal for:**
- Simple factual Q&A
- Fixed known pipeline
- Tight latency requirements
- Need deterministic behavior
- Very simple queries

### Architecture

```mermaid
graph TB
    A[User Query] --> B[Initial State]
    
    B --> C["💭 Thought:<br/>What do I need to do?"]
    
    C --> D{"🎯 Action Decision"}
    
    D -->|Need docs| E1["⚡ Action: Retrieve"]
    D -->|Need calc| E2["⚡ Action: Calculate"]
    D -->|Need search| E3["⚡ Action: Search"]
    D -->|Have answer| E4["⚡ Action: Finish"]
    
    E1 --> F1["👁️ Observe:<br/>Retrieved docs"]
    E2 --> F2["👁️ Observe:<br/>Calculation result"]
    E3 --> F3["👁️ Observe:<br/>Search results"]
    
    F1 --> G["Update State<br/>with observations"]
    F2 --> G
    F3 --> G
    
    G --> H{"Continue?"}
    
    H -->|Yes| C
    H -->|No| I["Generate Final Answer"]
    
    E4 --> I
    
    I --> J[Return Answer + Trace]
    
    style A fill:#e1f5ff
    style C fill:#fff9c4
    style D fill:#f3e5f5
    style E1 fill:#e8f5e9
    style E2 fill:#e8f5e9
    style E3 fill:#e8f5e9
    style F1 fill:#ffe0b2
    style F2 fill:#ffe0b2
    style F3 fill:#ffe0b2
    style I fill:#c8e6c9
    style J fill:#b3e5fc
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
import re
from typing import List, Dict, Optional
import time
from dataclasses import dataclass
from enum import Enum

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

# Expected output:
# ✓ Imports successful

## 2️⃣ Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
INDEX_NAME = 'react_rag_docs'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
LLM_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'  # Need Opus for reasoning

# ReAct Parameters
MAX_ITERATIONS = 10  # Maximum Thought-Action-Observation cycles
RETRIEVAL_TOP_K = 3  # Docs per retrieval action

print(f"Configuration:")
print(f"  Model: {LLM_MODEL.split('.')[-1][:40]}")
print(f"  Max iterations: {MAX_ITERATIONS}")
print(f"  Retrieval per action: Top-{RETRIEVAL_TOP_K}")

# Expected output:
# Configuration:
#   Model: claude-opus-4-1-20250805-v1:0
#   Max iterations: 10
#   Retrieval per action: Top-3

## 3️⃣ Initialize Services

In [ ]:
opensearch = OpenSearchManager(region=AWS_REGION, index_name=INDEX_NAME)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
llm = BedrockLLM(AWS_REGION, LLM_MODEL, temperature=0.7)

print("✓ Services initialized")

# Expected output:
# ✓ Services initialized

## 4️⃣ Load Knowledge Base

In [ ]:
sample_documents = [
    # AWS Bedrock models
    "Claude Opus 4.1: Best for complex reasoning, $15 input/$75 output per million tokens.",
    "Claude Sonnet: Balanced performance, $3 input/$15 output per million tokens.",
    "Claude Haiku: Most economical, $0.25 input/$1.25 output per million tokens.",
    
    # Performance characteristics
    "Opus: Highest quality, best for analysis and multi-step reasoning.",
    "Sonnet: Good balance of speed and quality for most tasks.",
    "Haiku: Fastest response time, suitable for simple queries.",
    
    # Cost optimization
    "Use Haiku for simple tasks: 20x cheaper than Opus.",
    "Use Sonnet for general-purpose: 5x cheaper than Opus.",
    "Use Opus only when complexity justifies the cost.",
    
    # RAG patterns
    "Simple RAG: Fast and cheap, good for straightforward queries.",
    "Self-RAG: Iterative improvement, 2-3x cost but better quality.",
    "Tree of Thoughts: Explores multiple paths, 3-5x cost.",
    "ReAct: Adaptive reasoning and acting, variable cost.",
    
    # Scaling considerations
    "At 10K queries/month with Haiku: ~$25/month.",
    "At 10K queries/month with Sonnet: ~$300/month.",
    "At 10K queries/month with Opus: ~$1,500/month.",
    
    # Best practices
    "Start with Simple RAG, add complexity only if needed.",
    "Monitor cost per query and adjust model/pattern accordingly.",
    "Cache embeddings to reduce repeated computation costs.",
    "Use adaptive routing to match query to optimal model.",
]

# Index documents
opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

print("Indexing knowledge base...")
documents = []
for i, text in enumerate(sample_documents):
    embedding = embedder.embed_text(text)
    documents.append({
        'text': text,
        'embedding': embedding,
        'metadata': {'doc_id': i}
    })

opensearch.index_documents(documents)
print(f"✓ Indexed {len(documents)} documents")

# Expected output:
# Indexing knowledge base...
# ✓ Indexed 21 documents

## 5️⃣ ReAct Framework

In [ ]:
class ActionType(Enum):
    RETRIEVE = "retrieve"
    CALCULATE = "calculate"
    FINISH = "finish"

@dataclass
class ReActStep:
    """One Thought-Action-Observation cycle"""
    iteration: int
    thought: str  # Reasoning about what to do
    action_type: ActionType  # Which action to take
    action_input: str  # Input for the action
    observation: str  # Result from action

print("✓ ReAct framework defined")

# Expected output:
# ✓ ReAct framework defined

## 6️⃣ Action Executors

In [ ]:
def execute_retrieve_action(query: str) -> str:
    """Execute retrieval action"""
    query_emb = embedder.embed_text(query)
    docs = opensearch.vector_search(query_emb, top_k=RETRIEVAL_TOP_K)
    
    if not docs:
        return "No documents found."
    
    result = "Retrieved:\n"
    for i, doc in enumerate(docs, 1):
        result += f"{i}. {doc['text']}\n"
    
    return result

def execute_calculate_action(expression: str) -> str:
    """Execute calculation action"""
    try:
        # Safe eval with limited scope
        allowed_names = {"__builtins__": {}}
        result = eval(expression, allowed_names)
        return f"Calculation result: {result}"
    except Exception as e:
        return f"Calculation error: {str(e)}"

print("✓ Action executors ready")

# Expected output:
# ✓ Action executors ready

## 7️⃣ ReAct Loop

In [ ]:
def react_rag(query: str, max_iterations: int = 10) -> Dict:
    """
    ReAct RAG: Reason and Act in interleaved cycles.
    
    Returns:
        Dict with answer, trace, metadata
    """
    start_time = time.time()
    
    print(f"Query: {query}\n")
    print("="*70)
    
    # Initialize
    trace = []
    scratchpad = f"Question: {query}\n\n"  # Running context
    
    for iteration in range(1, max_iterations + 1):
        print(f"\n{'='*70}")
        print(f"Iteration {iteration}/{max_iterations}")
        print(f"{'='*70}")
        
        # Thought: Reason about next action
        print("\n💭 Thought:")
        
        thought_prompt = f"""
You are solving this step-by-step using ReAct (Reason + Act).

{scratchpad}

Available Actions:
1. retrieve[query] - Search knowledge base
2. calculate[expression] - Evaluate math
3. finish[answer] - Return final answer

What should you do next?

Respond in this format:
Thought: [your reasoning about what to do next]
Action: [action_name[input]]

Examples:
Thought: I need to find pricing information for Claude Opus.
Action: retrieve[Claude Opus pricing]

Thought: Now I need to calculate the monthly cost.
Action: calculate[15 * 100000 / 1000000]

Thought: I have all the information needed to answer the question.
Action: finish[The answer is...]
"""
        
        response = llm.generate(thought_prompt, temperature=0.7)
        
        # Parse Thought and Action
        thought = ""
        action_type = None
        action_input = ""
        
        if "Thought:" in response:
            thought_start = response.find("Thought:") + len("Thought:")
            thought_end = response.find("Action:") if "Action:" in response else len(response)
            thought = response[thought_start:thought_end].strip()
            print(f"  {thought[:100]}...")
        
        if "Action:" in response:
            action_text = response[response.find("Action:") + len("Action:"):].strip()
            
            # Parse action
            if action_text.startswith("retrieve["):
                action_type = ActionType.RETRIEVE
                action_input = action_text[len("retrieve["):action_text.find("]")]
            elif action_text.startswith("calculate["):
                action_type = ActionType.CALCULATE
                action_input = action_text[len("calculate["):action_text.find("]")]
            elif action_text.startswith("finish["):
                action_type = ActionType.FINISH
                action_input = action_text[len("finish["):action_text.rfind("]")]
        
        if not action_type:
            print("  Could not parse action, assuming need to finish")
            action_type = ActionType.FINISH
            action_input = "Based on the context, here's my answer: " + response[:200]
        
        # Action: Execute
        print(f"\n⚡ Action: {action_type.value}")
        print(f"  Input: {action_input[:80]}...")
        
        observation = ""
        
        if action_type == ActionType.RETRIEVE:
            observation = execute_retrieve_action(action_input)
        elif action_type == ActionType.CALCULATE:
            observation = execute_calculate_action(action_input)
        elif action_type == ActionType.FINISH:
            # Done!
            final_answer = action_input
            
            print(f"\n✓ Finished after {iteration} iterations")
            
            total_time = time.time() - start_time
            
            return {
                'answer': final_answer,
                'trace': trace,
                'iterations': iteration,
                'completed': True,
                'metadata': {
                    'total_time': total_time,
                    'num_actions': len(trace)
                }
            }
        
        # Observation: Process results
        print(f"\n👁️  Observation:")
        print(f"  {observation[:150]}...")
        
        # Record step
        step = ReActStep(
            iteration=iteration,
            thought=thought,
            action_type=action_type,
            action_input=action_input,
            observation=observation
        )
        trace.append(step)
        
        # Update scratchpad
        scratchpad += f"Thought {iteration}: {thought}\n"
        scratchpad += f"Action {iteration}: {action_type.value}[{action_input}]\n"
        scratchpad += f"Observation {iteration}: {observation}\n\n"
    
    # Max iterations reached
    print(f"\n⚠️  Max iterations reached ({max_iterations})")
    
    total_time = time.time() - start_time
    
    return {
        'answer': "Could not reach final answer within iteration limit.",
        'trace': trace,
        'iterations': max_iterations,
        'completed': False,
        'metadata': {
            'total_time': total_time,
            'num_actions': len(trace)
        }
    }

print("✓ ReAct RAG pipeline ready")

# Expected output:
# ✓ ReAct RAG pipeline ready

## 8️⃣ Test ReAct RAG

In [ ]:
# Test with queries requiring different action types
test_queries = [
    "What's the monthly cost of running 10,000 queries with Claude Haiku?",
    "Should I use Opus or Sonnet for a production RAG system?",
]

results = []

for i, query in enumerate(test_queries, 1):
    print(f"\n{'#'*70}")
    print(f"# TEST {i}/{len(test_queries)}")
    print(f"{'#'*70}\n")
    
    result = react_rag(query, max_iterations=MAX_ITERATIONS)
    results.append(result)
    
    print("\n" + "="*70)
    print("REACT TRACE")
    print("="*70)
    
    for step in result['trace']:
        print(f"\n[{step.iteration}] 💭 Thought: {step.thought[:80]}...")
        print(f"    ⚡ Action: {step.action_type.value}[{step.action_input[:50]}...]")
        print(f"    👁️  Observation: {step.observation[:80]}...")
    
    print("\n" + "="*70)
    print("FINAL ANSWER")
    print("="*70)
    print(f"\n{result['answer'][:400]}...\n")
    print(f"Completed: {result['completed']}")
    print(f"Iterations: {result['iterations']}")
    print(f"Actions taken: {result['metadata']['num_actions']}")
    print(f"\n{'#'*70}\n")

# Expected output:
# [Shows interleaved Thought-Action-Observation cycles]

## 9️⃣ Comparison: ReAct vs Traditional Patterns

In [ ]:
comparison_query = "What's the cost difference between Haiku and Opus for 5000 queries?"

print("="*70)
print("REACT RAG (Adaptive Reasoning + Acting)")
print("="*70 + "\n")

react_result = react_rag(comparison_query, max_iterations=MAX_ITERATIONS)

print("\n" + "="*70)
print("SIMPLE RAG (Fixed Pipeline)")
print("="*70 + "\n")

# Simple RAG
simple_start = time.time()
query_emb = embedder.embed_text(comparison_query)
simple_docs = opensearch.vector_search(query_emb, top_k=5)
simple_context = [d['text'] for d in simple_docs]
simple_answer = llm.generate_with_context(comparison_query, simple_context)
simple_time = time.time() - simple_start

print(f"Retrieved {len(simple_docs)} docs")
print(f"Generated answer in {simple_time:.2f}s")

# Comparison
print("\n" + "="*70)
print("COMPARISON")
print("="*70)

print(f"\n🔄 Approach:")
print(f"  ReAct: Adaptive Thought-Action-Observation loops")
print(f"  Simple: Fixed Retrieve → Generate pipeline")

print(f"\n🎯 Actions Taken:")
print(f"  ReAct: {react_result['metadata']['num_actions']} actions")
action_types = [s.action_type.value for s in react_result['trace']]
print(f"    Types: {', '.join(set(action_types))}")
print(f"  Simple: 1 retrieve action (fixed)")

print(f"\n💭 Reasoning Visibility:")
print(f"  ReAct: Explicit thoughts at each step")
for i, step in enumerate(react_result['trace'][:3], 1):
    print(f"    {i}. {step.thought[:50]}...")
print(f"  Simple: No visible reasoning")

print(f"\n⏱️  Latency:")
print(f"  ReAct: {react_result['metadata']['total_time']:.2f}s ({react_result['iterations']} iterations)")
print(f"  Simple: {simple_time:.2f}s (1 pass)")

print(f"\n💰 Cost (estimated):")
react_cost = 0.05 * react_result['iterations']
simple_cost = 0.05
print(f"  ReAct: ~${react_cost:.3f} ({react_result['iterations']} LLM calls)")
print(f"  Simple: ~${simple_cost:.3f} (1 LLM call)")

print(f"\n🧠 Adaptability:")
print(f"  ReAct: Can change approach mid-execution")
print(f"  Simple: Fixed sequence, no adaptation")

print(f"\n📝 Answers (First 200 chars):\n")
print(f"ReAct:\n{react_result['answer'][:200]}...\n")
print(f"Simple RAG:\n{simple_answer[:200]}...")

print(f"\n💡 Key Advantage:")
print(f"  ReAct adapts its strategy based on what it learns,")
print(f"  rather than following a fixed pipeline.")

# Expected output:
# [Shows ReAct's adaptive behavior vs Simple RAG's fixed pipeline]

## 🔟 Analyze Action Patterns

In [ ]:
print("="*70)
print("ACTION PATTERN ANALYSIS")
print("="*70 + "\n")

for test_idx, (query, result) in enumerate(zip(test_queries, results), 1):
    print(f"Query {test_idx}: {query}")
    print(f"\n  Completed: {result['completed']}")
    print(f"  Iterations: {result['iterations']}")
    print(f"\n  Action Sequence:")
    
    for step in result['trace']:
        print(f"    {step.iteration}. {step.action_type.value}: {step.action_input[:50]}")
    
    # Action statistics
    action_counts = {}
    for step in result['trace']:
        action_type = step.action_type.value
        action_counts[action_type] = action_counts.get(action_type, 0) + 1
    
    print(f"\n  Action Distribution:")
    for action, count in action_counts.items():
        print(f"    {action}: {count}x")
    
    print()

print("="*70)
print("KEY INSIGHTS")
print("="*70 + "\n")

print("ReAct Behavior Patterns:")
print("  ✓ Adapts action sequence to query needs")
print("  ✓ Can mix different action types")
print("  ✓ Decides when enough information is gathered")
print("  ✓ Explicit reasoning at each decision point")
print("  ✓ Self-directed problem solving")

print("\nWhen ReAct Excels:")
print("  - Uncertain what actions are needed")
print("  - Requires multiple tool types")
print("  - Need adaptive behavior")
print("  - Complex multi-step problems")
print("  - Want visible reasoning trace")

# Expected output:
# [Analysis of action patterns and adaptive behavior]

## 1️⃣1️⃣ Summary & Key Takeaways

### What We Built

✅ Thought-Action-Observation loop
✅ Multiple action types (retrieve, calculate, finish)
✅ Adaptive decision-making
✅ Visible reasoning at each step
✅ Self-directed problem solving

### Performance Characteristics

- **Latency**: Variable (2-10+ iterations)
- **Cost**: 2-5x Simple RAG (multiple LLM calls)
- **Adaptability**: High (changes based on observations)
- **Transparency**: Full reasoning + action trace
- **Flexibility**: Not bound to fixed pipeline

### ReAct vs Other Patterns

| Aspect | Simple RAG | Chain of Thought | ReAct |
|--------|-----------|------------------|-------|
| Pipeline | Fixed | Pre-planned steps | Adaptive |
| Actions | 1 (retrieve) | Multiple (sequential) | Multiple (adaptive) |
| Reasoning | Hidden | Visible steps | Visible thoughts |
| Flexibility | None | Limited | High |
| Tool Use | Fixed | Pre-defined | Dynamic |
| Latency | ~2s | ~6-12s | ~4-12s |
| Cost | $0.05 | $0.10-0.20 | $0.10-0.25 |
| Best for | Simple Q&A | Complex reasoning | Adaptive tasks |

### When to Use ReAct

**Use ReAct when:**
- Don't know what actions needed upfront
- Need to adapt based on findings
- Require multiple tool types
- Want visible reasoning process
- Complex problem-solving tasks
- Research and exploration

**Skip ReAct when:**
- Simple queries with known pipeline
- Fixed retrieval always sufficient
- Tight latency requirements
- Need deterministic behavior
- Very tight budget

### Key Insights

1. **Interleaved Loop**: Reasoning and acting alternate
2. **Adaptive**: Decides next action based on observations
3. **Transparent**: See both thoughts and actions
4. **Flexible**: Not constrained to fixed sequence
5. **Self-directed**: Autonomous decision-making

### Best Practices

1. **Clear Action Set**: Define available actions clearly

2. **Action Format**: Use parseable format (action[input])

3. **Scratchpad**: Maintain running context

4. **Max Iterations**: Cap to prevent runaway loops

5. **Finish Action**: Clear termination condition

6. **Error Handling**: Handle failed actions gracefully

7. **Log Trace**: Keep full Thought-Action-Observation history

8. **Action Variety**: Support diverse action types

### Advanced Techniques

- **More Actions**: Add search, API calls, code execution
- **Action Chains**: Pre-learned action sequences
- **Parallel Actions**: Execute independent actions together
- **Action Confidence**: Score action selection confidence
- **Learned Policies**: Train which actions work best
- **Hierarchical ReAct**: High-level and low-level actions

### Limitations

1. **Unpredictable**: Hard to estimate iterations/cost
2. **Higher Cost**: Multiple reasoning rounds
3. **Longer Latency**: Sequential iterations
4. **Action Quality**: Depends on LLM's decisions
5. **Infinite Loops**: Risk if termination unclear

### Real-world Applications

**Where ReAct Excels:**
- Research assistants (adaptive exploration)
- Data analysis (mix queries + calculations)
- Technical support (diagnose → retrieve → fix)
- Interactive tutoring (adapt to student needs)
- Complex problem solving (multi-tool tasks)

### Cost Breakdown (per query)

**4-iteration example:**
- Iteration 1 (thought + action): $0.05
- Iteration 2 (thought + action): $0.05
- Iteration 3 (thought + action): $0.05
- Iteration 4 (finish): $0.03
- **Total**: ~$0.18 (vs $0.05 Simple RAG)

**Worth it?** When need adaptability: 3.6x cost → flexible problem solving

### ReAct in the Pattern Landscape

**ReAct vs Agentic RAG:**
- Similar: Both autonomous, both use tools
- Different: ReAct has explicit Thought-Action-Observation structure

**ReAct vs Chain of Thought:**
- CoT: Pre-planned steps, sequential
- ReAct: Adaptive steps, interleaved reasoning/acting

**Combined:** ReAct + Tree of Thoughts = explore multiple ReAct paths

### Action Diversity

ReAct can use any action:
- **Retrieval**: Search knowledge base
- **Calculation**: Math operations
- **Search**: Web/database queries
- **API Calls**: External services
- **Code Execution**: Run programs
- **Tool Use**: Any available tool

### Next Steps

- **Expand Actions**: Add more tool types
- **Optimize Routing**: Learn which actions work best
- **Parallel ReAct**: Multiple agents in ReAct loops
- **Memory**: Remember past ReAct sessions
- **User Interaction**: Ask user during loop

---

## 🎉 Phase 2 Progress!

**Progress**: 17/37 patterns (46%)
- Phase 1: 10/10 ✅ Complete
- Phase 2: 7/12 ✅ Multi-modal + Agentic + CRAG + Self-RAG + Tree + CoT + ReAct

**Remaining Phase 2**: 5 patterns (LangGraph, Memory, Ensemble, Iterative, Few-shot)

## 🧹 Cleanup

In [ ]:
# Uncomment to delete index
# opensearch.delete_index(INDEX_NAME)
# print(f"✓ Deleted index: {INDEX_NAME}")

print("\nTo delete the index later:")
print(f"  opensearch.delete_index('{INDEX_NAME}')")

# Expected output:
# 
# To delete the index later:
#   opensearch.delete_index('react_rag_docs')